# 49. Three neural variants, because the neural direction is the one with room

**One variable against ledger row 106** (`neural_fixed`, CV 0.965402): the input frame for the
first arm, the architecture for the other two. Same folds, same seed, same encoder fingerprinted
`0642e41750ef8bab`, same 30-epoch OneCycle schedule taking the final epoch with no validation
consulted.

| arm | one variable against | what changes |
|---|---|---|
| `neural_fe` | row 106 `neural_fixed` | the **ratio block** added to the input |
| `neural_wide` | `neural_fe`, in this kernel | **width and depth**, 1024/512/256 |
| `neural_res` | `neural_fe`, in this kernel | **residual blocks** instead of a plain stack |

## Why neural and not something else

The last three gates say this plainly. Gate 3 offered the combiner seven candidates including four
model families it had never contained, and **the only single that cleared the floor was
`neural_lookup`** at +0.000078, from the worst-scoring model in the ledger. The four new families,
including the most decorrelated vector this repo has ever produced, were worth +0.000034 between
them. Gate 4 added the best single model here and got +0.000021.

Meanwhile all four neural vectors carry real weight in the 48-member combiner: `neural_lookup`
+0.0758, `neural_te` +0.0283, `neural` +0.0246. Boosted trees are saturated in this stack. Neural
nets are the one family where a new member has recently been worth something, and there are only
three of them against forty-odd trees.

**And the obvious gap: no neural here has ever seen the ratio block.** It is worth +0.000356 to
+0.000906 to every tree family, and rows 90 to 93 and 108 to 109 established that it carries
cross-column information 1-way target encoding cannot represent. A network is if anything better
placed to use it than an axis-aligned tree.

## What is NOT being tried, and why

Target-encoding the ratio columns. It looks attractive and I think it is a trap. Three independent
routes to per-value structure have now returned null: the decimal digit at -0.000132, six times the
bin resolution at +0.000049, and per-value embeddings at -0.004221. A ratio column has 8,000 to
12,000 distinct values, so encoding it is per-value structure on a **derived** column, which is
further from the generator than any of the three that already failed. Logged rather than run.

## The prediction, written before the run

`neural_fe` above row 106 by **+0.0005 to +0.0025**. The two architecture arms within +/-0.0010 of
`neural_fe`, because width and residual connections are knobs and this repo's knobs do not pay.

I expect **at least one arm to fire in the gate**, and I expect the gate value to come more from
*disagreement between the three* than from any one being strong.

**The honest case against**, which has now beaten my predictions five runs running. The neural
family may be carrying weight because it is *one* different thing, not because more of it helps.
Three more networks trained on the same folds with the same schedule may be three more copies of
the same disagreement, in which case they will split `neural_te`'s weight the way `xgb_tuned` split
XGBoost's in row 126 and the gate will return twenty millionths.

## What this decides

Nothing about the stack. Membership is a separate notebook and a separate ledger row.

In [ ]:
SMOKE = True

SEED = 42
N_SPLITS = 5
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# Row 106's architecture and schedule. Only the arm-specific change moves.
EMB_DIM = 8
HIDDEN = (512, 256, 128)
WIDE_HIDDEN = (1024, 512, 256)
DROPOUT = 0.2
EPOCHS, BATCH, LR, WD = 30, 4096, 3e-3, 1e-4
USE_MISSING_MASK = True

ARMS = ["neural_fe", "neural_wide", "neural_res"]

ROW106_CV = 0.965402
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"

if SMOKE:
    EPOCHS, N_SPLITS = 2, 2

print(f"SMOKE = {SMOKE}   arms {ARMS}   epochs {EPOCHS}")
print("no validation is consulted at any point; the final epoch is taken")

In [ ]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")

## The encoder, fingerprinted against 13

In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")

In [ ]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## The ratio block, and the three input frames

The 13 composition columns from notebook 40, unchanged and added raw. They go through the same
quantile transform as the rest of the numeric block, fit on training rows only inside the fold.

In [ ]:
DST, SM_, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                    "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM_, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM_], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm

RB_TR = ratio_block(train).to_numpy(np.float32)
RB_TE = ratio_block(test).to_numpy(np.float32)
print(f"row 106 fed {len(NUM_COLS) + len(ENC_COLS)} numeric + {len(NUM_COLS)} mask columns")
print(f"these arms feed {len(NUM_COLS) + len(ENC_COLS) + len(RATIO_COLS)} numeric "
      f"+ {len(NUM_COLS)} mask")

cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)
Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)

In [ ]:
class TabMLP(nn.Module):
    """Row 106's network, with the hidden sizes as a parameter."""

    def __init__(self, n_num, cat_sizes, emb_dim=EMB_DIM, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim) for s in cat_sizes])
        dim = n_num + emb_dim * len(cat_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat([xn] + e, dim=1)).squeeze(1)


class ResBlock(nn.Module):
    def __init__(self, d, p):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d, d), nn.BatchNorm1d(d), nn.SiLU(),
                               nn.Dropout(p), nn.Linear(d, d), nn.BatchNorm1d(d))
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(x + self.f(x))


class TabResNet(nn.Module):
    """Same width and depth budget as row 106, but the hidden stack is residual. A
    different optimisation geometry rather than a bigger model."""

    def __init__(self, n_num, cat_sizes, emb_dim=EMB_DIM, d=256, blocks=3, p=DROPOUT):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim) for s in cat_sizes])
        self.inp = nn.Sequential(
            nn.Linear(n_num + emb_dim * len(cat_sizes), d), nn.BatchNorm1d(d), nn.SiLU())
        self.body = nn.Sequential(*[ResBlock(d, p) for _ in range(blocks)])
        self.head = nn.Linear(d, 1)

    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.head(self.body(self.inp(torch.cat([xn] + e, dim=1)))).squeeze(1)


def make_model(arm, n_num):
    if arm == "neural_wide":
        return TabMLP(n_num, cat_sizes, hidden=WIDE_HIDDEN)
    if arm == "neural_res":
        return TabResNet(n_num, cat_sizes)
    return TabMLP(n_num, cat_sizes, hidden=HIDDEN)


NUM_WORKERS = 0 if os.name == "nt" else 2


def make_loader(xn, xc, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(
        ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEV.type == "cuda"), drop_last=drop_last)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xn, xc, _ in loader:
        out.append(torch.sigmoid(model(xn.to(DEV), xc.to(DEV))).float().cpu().numpy())
    return np.concatenate(out)


LOG = OUT / "49_neural_variety.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, device={DEV}, arms={ARMS} ===")

In [ ]:
from sklearn.preprocessing import QuantileTransformer

results = {}
t_start = time.time()
for arm in ARMS:
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    per = []
    t_arm = time.time()
    for f in range(N_SPLITS):
        seed_all(SEED + f)
        tr_i = np.where(folds != f)[0]
        va_i = np.where(folds == f)[0]
        Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
        num_tr = np.hstack([Etr[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[tr_i]])
        num_va = np.hstack([Eva[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[va_i]])
        num_te = np.hstack([Ete[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TE])

        med = np.nanmedian(num_tr, axis=0)
        med = np.where(np.isnan(med), 0.0, med)

        def prep(a):
            return np.where(np.isnan(a), med, a)

        qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                                 subsample=200_000, random_state=SEED).fit(prep(num_tr))

        def finish(a, m):
            x = qt.transform(prep(a)).astype(np.float32)
            return np.hstack([x, m]) if USE_MISSING_MASK else x

        Xn_tr = finish(num_tr, mask_tr[tr_i])
        Xn_va = finish(num_va, mask_tr[va_i])
        Xn_te = finish(num_te, mask_te)

        tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], y[tr_i], BATCH, True, drop_last=True)
        va_loader = make_loader(Xn_va, Xc_tr[va_i], None, BATCH * 4, False)
        te_loader = make_loader(Xn_te, Xc_te, None, BATCH * 4, False)

        model = make_model(arm, Xn_tr.shape[1]).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr_loader))
        lossf = nn.BCEWithLogitsLoss()
        for ep in range(EPOCHS):
            model.train()
            for xn, xc, yy in tr_loader:
                opt.zero_grad(set_to_none=True)
                loss = lossf(model(xn.to(DEV), xc.to(DEV)), yy.to(DEV))
                loss.backward()
                opt.step()
                sched.step()
            if ep % 10 == 0 or ep == EPOCHS - 1:
                print(f"  {arm} fold {f} epoch {ep:>2}: "
                      f"val AUC {roc_auc_score(y[va_i], predict(model, va_loader)):.6f}")
        oof[va_i] = predict(model, va_loader)
        test_pred += predict(model, te_loader) / N_SPLITS
        per.append(float(roc_auc_score(y[va_i], oof[va_i])))
        del model
        note(f"{arm} fold {f}: AUC {per[-1]:.6f}   elapsed {(time.time()-t_start)/60:.1f} min")
    results[arm] = {"oof": oof, "test": test_pred, "per": np.array(per)}
    note(f"{arm}: CV {np.mean(per):.6f} +/- {np.std(per):.6f} "
         f"in {(time.time()-t_arm)/60:.1f} min")
print(f"\nall arms done in {(time.time()-t_start)/60:.1f} min")

In [ ]:
def load_saved(stem):
    for name in (f"{stem}_oof.npy", f"{stem}.npy"):
        try:
            return np.load(locate(name))
        except FileNotFoundError:
            continue
    return None


print(f"{'arm':13}{'CV':>11}{'sd':>10}{'vs row 106':>13}")
for a in ARMS:
    p = results[a]["per"]
    print(f"{a:13}{p.mean():11.6f}{p.std():10.6f}{p.mean() - ROW106_CV:+13.6f}")

base = load_saved("neural_fixed")
if base is not None and not SMOKE:
    bp = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(N_SPLITS)])
    print(f"\nrow 106 re-scored here: {bp.mean():.6f}, ledger {ROW106_CV:.6f}, "
          f"delta {bp.mean() - ROW106_CV:+.2e}")
    d = results["neural_fe"]["per"] - bp
    sd = d.std(ddof=1)
    print(f"neural_fe vs row 106, one variable (the ratio block): {d.mean():+.6f}, "
          f"sd {sd:.6f}, {int((d > 0).sum())}/{N_SPLITS} folds"
          + (f", t={d.mean()/(sd/np.sqrt(N_SPLITS)):.2f}" if sd > 0 else ""))
    for a in ("neural_wide", "neural_res"):
        da = results[a]["per"] - results["neural_fe"]["per"]
        s2 = da.std(ddof=1)
        print(f"{a} vs neural_fe, one variable (the architecture): {da.mean():+.6f}, "
              f"sd {s2:.6f}, {int((da > 0).sum())}/{N_SPLITS} folds")

print("\nDISAGREEMENT, which is what the gate will actually price:")
for a in ARMS:
    row = []
    for b in ARMS + ["neural_fixed", "neural_te", "neural_lookup"]:
        v = results[b]["oof"] if b in results else load_saved(b)
        if v is None or b == a:
            continue
        # Saved vectors are full length; under SMOKE this run is a subsample, so they are
        # indexed back through ROW_IDX rather than compared against a different length,
        # which pandas would align on index and silently return nonsense for.
        if b not in results:
            v = v[ROW_IDX]
        row.append(f"{b} {pd.Series(results[a]['oof']).corr(pd.Series(v), method='spearman'):.4f}")
    print(f"  {a:13} " + "  ".join(row))

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for a in ARMS:
    np.save(OUT / f"{pre}{a}_oof.npy", results[a]["oof"])
    np.save(OUT / f"{pre}{a}_test.npy", results[a]["test"])
    print(f"wrote {pre}{a}_oof.npy, {pre}{a}_test.npy")
print("\nledger lines:")
for a in ARMS:
    p = results[a]["per"]
    print(f"  name    {a}\n  cv_mean {p.mean():.6f}\n  cv_std  {p.std():.6f}")
print(f"\n  encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"ratio block pure {BLOCK_OK}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is a separate notebook and a separate ledger row.")